In [ ]:
import logging

import matplotlib.pyplot as plt
import numpy as np
from math import ceil
import scipy.signal as ss

# Set up logging configuration
logging.basicConfig(
    format="[%(asctime)s] - [%(name)s]\t%(levelname)s\t%(message)s",
    handlers=[
        logging.StreamHandler(),
        logging.FileHandler("tinyprobe.log"),
    ],
)

In [ ]:
from tinyprobe.protocol.commands import TinyProbeCmdSeq, SwitchSpiMux, ControlPower, SleepMs, TriggerShot, WriteFPGAReg
from tinyprobe.executor import TPExecutor, TPExecSend, TPExecNOP, TPExecRecv, TPExecSave

# from tinyprobe.comlink.wifi4 import TPComWiFi4
from tinyprobe.comlink.wifi6 import TPComWiFi6

from tinyprobe.drivers.hal.fpga import TP_HAL_FPGA
from tinyprobe.drivers.hal.tx7332 import TP_HAL_TX7332
from tinyprobe.drivers.hal.afe5832lp import TP_HAL_AFE5832LP

In [ ]:
# Auto reload imports
%load_ext autoreload
%autoreload 2

### Establish Connection

In [ ]:
IPADDR = "192.168.50.144"

# com_link = TPComWiFi4(IPADDR)
com_link = TPComWiFi6(IPADDR)
com_link.open()

if not com_link.ping():
    raise Exception("Ping failed")

print("Connected to", com_link.name)

### Instantiate  Command Executor

In [ ]:
executor = TPExecutor(log=logging.DEBUG)
executor.set_com_link(com_link)

### Parameters for measurements

In [ ]:
from tinyprobe.drivers.hal.fpga import FPGA_MAX_LVDS_LANES

# Settings
afe_clk_high_speed = True
desired_fps = 200
read_depth_shallow=False
full_ch_array = True

# Derived parameters
meas_period_ms=1/desired_fps * 1000
n_shots = 150

if read_depth_shallow:
    fifo_read_depth = 400
else:
    fifo_read_depth = 2048

if full_ch_array:
    en_lvds_ch = [x for x in range(FPGA_MAX_LVDS_LANES)]
else:
    en_lvds_ch = [10]

### FPGA settings

In [ ]:
n_packets_to_read = ceil(fifo_read_depth * 20 * len(en_lvds_ch) /8/ 1000)

fpga = TP_HAL_FPGA()

afe_start_capt_delay_us = 0

cmd_seq_fpga = fpga.default_config(active_lvds_lanes=en_lvds_ch,
                                   sensing_depth_samples=fifo_read_depth,
                                   meas_period_ms=meas_period_ms,
                                   n_acquisitions=n_shots,
                                   afe_clk_high_speed=afe_clk_high_speed,
                                   tx_bf_clk_high_speed=True,
                                   fpga_core_clk_high_speed=False,
                                   afe_start_capt_delay_us=afe_start_capt_delay_us)


# fpga.set_out_trigger_src(source="fifo_wr_done")
fpga.set_mcu_interrupt_src(source="tx_bf_sync")
                           
cmd_seq_fpga.extend(fpga.get_cmd_sequence(from_history=True))

executor.add(TPExecSend(cmd_seq=cmd_seq_fpga))

#### TX chip settings: Prepare Delays for PWs

In [ ]:
from pybf.pybf.transducer import Transducer

pulse_freq = 5.0*10**6

# VERMON NDT 
# Transucer settings
F_CENTRAL = pulse_freq
X_ELEM = 32
X_PITCH = 0.0006
X_WIDTH = 0
Y_ELEM = 1
Y_PITCH = 0
Y_WIDTH = 0

trans = Transducer(num_of_x_elements=X_ELEM,
                   num_of_y_elements=Y_ELEM,
                   x_pitch=X_PITCH,
                   y_pitch=Y_PITCH,
                   x_width=X_WIDTH,
                   y_width=Y_WIDTH,
                   f_central_hz=F_CENTRAL,
                   bandwidth_hz=F_CENTRAL,
                   active_elements=None)

tx_strategy = ["PW_1_0", 1, 0]
# tx_strategy = ["PW_15_15", 15, 15]
# tx_strategy = ["PW_5_15", 5, 15]


elements_coords = trans.elements_coords

# For CIRS Phantom
# speed_of_sound_global = 1540
speed_of_sound_global = 1000
# speed_of_sound_global = 1080

def calc_tx_delays(tx_strategy,
                   elements_coords,
                   speed_of_sound=speed_of_sound_global):
    
    # Step 0: Extract parameters
    num_of_pw = tx_strategy[1]
    max_angle = tx_strategy[2]
    
    # Step 1: Subtract minimum value
    el_coords_x = elements_coords[0,:] - elements_coords[0,:].min()

    # Step 2: Translate angles
    pw_angles_rad = np.radians(np.linspace(-max_angle, max_angle, num_of_pw)).reshape(-1,1)

    # Step 3: Calculate delays
    tx_delays = np.multiply(el_coords_x, np.sin(np.abs(pw_angles_rad)))/speed_of_sound

    # Swap the sign of the delays  for negative angles
    neg_ang_mask = np.sign(pw_angles_rad.flatten()) == -1
    tx_delays[neg_ang_mask, :] = tx_delays[neg_ang_mask, ::-1]

    return tx_delays, pw_angles_rad
    


tx_delays, pw_angles_rad = calc_tx_delays(tx_strategy, 
                                          elements_coords, 
                                          speed_of_sound=speed_of_sound_global)

In [ ]:
# plt.figure()

# for angle_id in range(len(pw_angles_rad)):
#     plt.plot(tx_delays[angle_id, :], label=str(pw_angles_rad[angle_id]))

# plt.legend()
# plt.gca().invert_yaxis()
# plt.show()

In [ ]:
TX_DEL_BITWIDTH = 12
BF_CLK_DIV_MAX = 5

tx_delays_clk_cycles = tx_delays*fpga.tx_bf_clk

tx_max_delay = np.max(tx_delays)*10**6

print("Max delay us: ", tx_max_delay)

if np.max(tx_delays_clk_cycles) >  2**TX_DEL_BITWIDTH:
    bf_clk_div = np.ceil(np.log2(np.max(tx_delays_clk_cycles) / 2**TX_DEL_BITWIDTH))
else:
    bf_clk_div = 0
    
print("BF clock divider: ", bf_clk_div)

if bf_clk_div > BF_CLK_DIV_MAX or bf_clk_div < 0:
    print("Error! Calculated clock divider exceeds the HW specs. Change the TX BF clock frequency!")

### Prepare the global TX chip settings

In [ ]:
tx_start_delay_us=5
tr_sw_delay_us=1.1

tx = TP_HAL_TX7332(bf_clk=fpga.tx_bf_clk)

tx.set_bf_clk_divider(factor=bf_clk_div)

cmd_seq_tx = tx.default_config(n_pulses=2, 
                               pulse_freq_hz=pulse_freq,
                               tx_start_delay_us=tx_start_delay_us,
                               tr_sw_delay_us=tr_sw_delay_us)

# com_link.send_cmd_seq(cmd_seq_tx, sleep_s=1)

executor.add(TPExecSend(cmd_seq=cmd_seq_tx))
executor.add(TPExecNOP(delay_ms=1000))

#### Program Delays

In [ ]:
# For Vermon NDT
RX_MAPPING = np.array([29, 9, 27, 7, 25, 4, 23, 11, 21, 5, 26, 3, 30, 1, 28, 0, 31, 2, 17, 15, 16, 14, 19, 13, 18, 12, 20, 10, 22, 8, 24, 6])
TX_MAPPING = np.array([16, 12, 0, 14, 2, 10, 4, 30, 6, 28, 8, 26, 22, 24, 18, 20, 19, 21, 23, 25, 7, 27, 1, 29, 3, 31, 9, 5, 11, 17, 13, 15])

tx_delays_clk_cycles = np.round(tx_delays_clk_cycles)

cmd_seq_tx = TinyProbeCmdSeq([SwitchSpiMux(code=3)])


for del_prof_id in range(tx_delays.shape[0]):
    tx.set_delays_s(tx_delays[del_prof_id][TX_MAPPING], dp_id=del_prof_id)

    # Send when 4 delay tables are accumulated
    if (del_prof_id % 4 == 0) & (del_prof_id != 0):
        cmd_seq_tx.extend(tx.get_cmd_sequence(from_history=True))
        com_link.send_cmd_seq(cmd_seq_tx, sleep_s=1)

        cmd_seq_tx = TinyProbeCmdSeq([SwitchSpiMux(code=3)])


tx.set_delay_prof_id(dp_id=int(tx_delays.shape[0] + 1) - 1)

cmd_seq_tx.extend(tx.get_cmd_sequence(from_history=True))
tx.load_profile()
cmd_seq_tx.extend(tx.get_cmd_sequence())

# com_link.send_cmd_seq(cmd_seq_tx, sleep_s=1)

executor.add(TPExecSend(cmd_seq=cmd_seq_tx))
executor.add(TPExecNOP(delay_ms=1000))

### Prepare AFE settings

In [ ]:
# Initialize AFE HAL
afe = TP_HAL_AFE5832LP()

# === Apply Fixed Gain ===
afe.set_fixed_gain(200)

# TODO: Check why this intermediate send is required to work

cmd_seq_afe_1 = afe.get_cmd_sequence(from_history=True)
executor.add(TPExecSend(cmd_seq=cmd_seq_afe_1))

# === Enable Test Pattern (Optional) ===
afe.set_test_pattern("normal")  # Other options: "ramp", "all_ones", etc.

# === TGC External Non-uniform Mode ===
# tgc_slope = 0.2598 * 4  # Gain increase per microsecond
tgc_slope = 0.2598 * 8  # Gain increase per microsecond
gain_step_period_us = afe.configure_tgc(start_db=0, stop_db=30, slope_db_per_us=tgc_slope)


# === Finalize and send configuration ===
cmd_seq_afe_2 = afe.get_cmd_sequence(from_history=True)
executor.add(TPExecSend(cmd_seq=cmd_seq_afe_2))
# com_link.send_cmd_seq(cmd_seq)

# === FPGA TGC Configuration ===
cmd_seq_fpga_2 = fpga.set_tgc_settings(gain_step_period_us=gain_step_period_us,
                                     start_delay_us=2,
                                     capture_time_us=200,
                                     finish_delay_us=10,
                                     match_with_wavegen=False)

# com_link.send_cmd_seq(cmd_seq_fpga_2)

executor.add(TPExecSend(cmd_seq=cmd_seq_fpga_2))

# === Analog Gain and Filtering Settings ===
afe.set_low_power_mode()         # or: afe.set_low_noise_mode()
afe.set_lna_gain(21)             # in dB
afe.set_pga_gain(27)             # in dB
afe.set_lpf_freq(15)            # in MHz
afe.set_lna_hpf_freq(310)       # will raise if unsupported

# === Enable only desired channels ===
# enabled_channels = [i for i in range(16)]  # example
afe.enable_channel_groups(en_lvds_ch)

# === Finalize and send configuration ===
cmd_seq_afe_3 = afe.get_cmd_sequence(from_history=True)
executor.add(TPExecSend(cmd_seq=cmd_seq_afe_3))
# com_link.send_cmd_seq(cmd_seq)

In [ ]:
executor.execute()

# Main Acquisition cells

#### Construct a Measurement Sequence

In [ ]:
# Use all the programmed delays (Apply them one by one to form a frame)
tx_delays_active = [x for x in range(len(tx_delays))]

# OR instead specify which delays to use to forma a frame
# tx_delays_active = [0]

dc_dc_off_delay_us = tx_start_delay_us + tr_sw_delay_us + tx_max_delay
read_fifo_delay_us = 3 +  fifo_read_depth/fpga.afe_clk * 1e6 - afe_start_capt_delay_us - dc_dc_off_delay_us

### Calculate trig package id (to power up AFE from global powerdown)

In [ ]:
MAX_THROUGHPUT = 32.4 * 1e6

AFE_SLEEP_TIME_MS = 5
trig_pack_id = int(n_packets_to_read - np.ceil(AFE_SLEEP_TIME_MS/1e3 /(com_link.packet_size*8/MAX_THROUGHPUT)))

if trig_pack_id < 0:
    trig_pack_id = 65535
    print("Warning: the time between the shots is too short to enable AFE/LVDS power down")

### Construct the sequence

In [ ]:
shot_delay_ms = 0
n_frames = 1


cmd_list=[]

## Exit power save modes

# Enable DC-DCs for TX chip
cmd_list.append(ControlPower(domain_id=1, enable=True)) # HV+
cmd_list.append(ControlPower(domain_id=2, enable=True)) # HV-
cmd_list.append(ControlPower(domain_id=3, enable=True)) # - 5

####

cmd_list.append(SwitchSpiMux(code=1)) # FPGA
cmd_list.append(ControlPower(domain_id=0, enable=True)) # Enable LVDS IO bank 
cmd_list.append(WriteFPGAReg(addr=10, val=16)) # Disable AFE global Power Down
cmd_list.append(SleepMs(delay=4))

cmd_seq = TinyProbeCmdSeq(cmd_list)

# If basically, we have one acquisition per frame 
if len(tx_delays_active) == 1:
    
    # Switch to TX chip
    cmd_seq.extend(TinyProbeCmdSeq([SwitchSpiMux(code=3)]))
                   
    # Change ID of the TX DEL profile
    tx.set_delay_prof_id(dp_id=tx_delays_active[0])
    cmd_seq.extend(tx.get_cmd_sequence(from_history=True))
    tx.load_profile()
    cmd_seq.extend(tx.get_cmd_sequence())
    
    # Switch to FPGA
    cmd_seq.extend(TinyProbeCmdSeq([SwitchSpiMux(code=1)]))
    # Trigger shot
    cmd_seq.extend(TinyProbeCmdSeq([TriggerShot(n_shots=n_shots, 
                                                n_packets=n_packets_to_read,
                                                dc_dc_off_delay_us=dc_dc_off_delay_us,
                                                read_fifo_delay_us=read_fifo_delay_us)]))

else:
    # Iterate over shots
    for shot_id in range(n_frames):
        # Iterate over active TX delays
        for id in tx_delays_active:
            # Switch to TX chip
            cmd_seq.extend(TinyProbeCmdSeq([SwitchSpiMux(code=3)]))
                           
            # Change ID of the TX DEL profile
            tx.set_delay_prof_id(dp_id=id)
            cmd_seq.extend(tx.get_cmd_sequence(from_history=True))
            tx.load_profile()
            cmd_seq.extend(tx.get_cmd_sequence())
        
            # Switch to FPGA
            cmd_seq.extend(TinyProbeCmdSeq([SwitchSpiMux(code=1)]))

            if shot_delay_ms >= 0:
                # Very large but not reachable
                trig_pack_id = 65534
            
            # Trigger shot
            cmd_seq.extend(TinyProbeCmdSeq([TriggerShot(n_shots=n_shots, 
                                                        n_packets=n_packets_to_read,
                                                        dc_dc_off_delay_us=dc_dc_off_delay_us,
                                                        read_fifo_delay_us=read_fifo_delay_us,
                                                        trig_pack_id=trig_pack_id)]))

            if shot_delay_ms >= 0:
                cmd_seq.extend(TinyProbeCmdSeq([SleepMs(delay=shot_delay_ms)]))
                cmd_seq.extend(TinyProbeCmdSeq([ControlPower(domain_id=0, enable=True)])) # Enable 2.5 V LVDS
                cmd_seq.extend(TinyProbeCmdSeq([WriteFPGAReg(addr=10, val=16)]))  # Disable AFE Power Down
                cmd_seq.extend(TinyProbeCmdSeq([SleepMs(delay=AFE_SLEEP_TIME_MS)]))

cmd_list=[]

## Enter power save modes

cmd_list.append(ControlPower(domain_id=0, enable=False)) # Disable LVDS IO bank 
cmd_list.append(WriteFPGAReg(addr=10, val=48)) # Enable AFE global Power Down

# Disable DC-DCs for TX chip
cmd_list.append(ControlPower(domain_id=1, enable=False)) # HV+
cmd_list.append(ControlPower(domain_id=2, enable=False)) # HV-
cmd_list.append(ControlPower(domain_id=3, enable=False)) # - 5

####

cmd_seq.extend(TinyProbeCmdSeq(cmd_list))

# Calculate shots
n_shots_rx = 0
for cmd in cmd_seq.cmd_list:
    if isinstance(cmd, TriggerShot):
        n_shots_rx += cmd.n_shots

print("Total shots to receive: ", n_shots_rx)


### Acquire the data

In [ ]:
# for _ in range(10):
executor.add(TPExecSend(cmd_seq=cmd_seq))
executor.add(TPExecRecv(n_packets=n_packets_to_read, n_shots=n_shots_rx, print_stats=True))

In [ ]:
result = executor.execute(keep_queue=True)

In [ ]:
print(f"\nFramerate:  {n_shots_rx/result[1][1]:5.2f} FPS")
print(f"Throughput: {result[1][0] * 8/1e6/result[1][1]:5.2f} Mbps")

### Save data

In [ ]:
from pathlib import Path

executor._command_queue.queue.clear()

filename = "acquisition.npy"
if Path(filename).exists():
    print(f"File {filename} already exists!")
    raise Exception("File already exists!")

executor.add(TPExecSave(filename=filename))

In [ ]:
executor.execute()

In [ ]:
# # Load from test.npy
# bytes_total = np.load("data/test.npy").tobytes()

# Get from executor
bytes_total = executor._data

In [ ]:
from IPython.display import clear_output

from tinyprobe.bytestream_parser import parse_bitstream

rf_data_list = []

bytes_per_shot = int(len(bytes_total)/n_shots_rx)

for i in range(n_shots_rx):

    rf_data = parse_bitstream(bytes_total[i*bytes_per_shot:(i+1)*bytes_per_shot], 
                              dbg_msgs = False,  
                              bit_slip_arr = [-2 for x in range(16)], 
                              read_size_samples = fifo_read_depth,
                              n_activ_ch = len(en_lvds_ch) * 2,
                              bits_per_sample=10)

    rf_data_list.append(rf_data)

    clear_output(wait=True)

print("Finished")

In [ ]:
channels_list = [i for i in range(len(en_lvds_ch) * 2)]

In [ ]:
from tinyprobe.visualization import plot_channels, plot_trace
import plotly.io as pio

pio.renderers.default = 'browser'

plot_channels(rf_data_list[49][RX_MAPPING, :].T, 18, channels_list)

In [ ]:
plot_trace(rf_data_list[20][RX_MAPPING, :].T, 18, fpga.afe_clk)

In [ ]:
# Filter specifications
fs = fpga.afe_clk  # Sampling frequency: 10 MHz
f_pass = [0.55*pulse_freq, 1.45*pulse_freq]  # Passband: 1 MHz to 4 MHz
f_stop = [0.55*pulse_freq-0.2e6, 1.45*pulse_freq+0.2e6]  # Stopband: 0.8 MHz to 4.2 MHz
gpass = 1  # Maximum loss in the passband (dB)
gstop = 60  # Minimum attenuation in the stopband (dB)

# Normalize frequencies to the Nyquist frequency
nyq = fs / 2
wp = np.array(f_pass) / nyq
ws = np.array(f_stop) / nyq

# Design the Chebyshev type II filter
b, a = ss.iirfilter(N=5, Wn=wp, rs=gstop, btype='band', analog=False, ftype='cheby2', output='ba')

# Frequency response
w, h = ss.freqz(b, a, worN=8000)

# # Plot the frequency response
# plt.figure()
# plt.plot((fs * 0.5 / np.pi) * w, 20 * np.log10(abs(h)), 'b')
# plt.title('Chebyshev Type II Bandpass Filter Frequency Response')
# plt.xlabel('Frequency (Hz)')
# plt.ylabel('Gain (dB)')
# plt.ylim(-80, 5)
# plt.grid()
# plt.show()

# Verify the specifications
passband = (f_pass[0] <= w * fs / (2 * np.pi)) & (w * fs / (2 * np.pi) <= f_pass[1])
stopband_lower = (w * fs / (2 * np.pi) <= f_stop[0])
stopband_upper = (w * fs / (2 * np.pi) >= f_stop[1])

print("Passband attenuation (dB):", np.max(20 * np.log10(abs(h[passband]))))
print("Stopband attenuation lower (dB):", np.min(20 * np.log10(abs(h[stopband_lower]))))
print("Stopband attenuation upper (dB):", np.min(20 * np.log10(abs(h[stopband_upper]))))

In [ ]:
numpy_arr = np.array(rf_data_list)
numpy_arr_filt = ss.filtfilt(b, a, numpy_arr)

In [ ]:
from pybf.pybf.image_settings import ImageSettings

image_x_range = [-0.03, 0.03]
image_z_range = [0.00, 0.025]

img_res = [600, 600]

db_range = 80

LATERAL_PIXEL_DENSITY_DEFAULT = 5

img_config = ImageSettings(image_x_range[0],
                           image_x_range[1],
                           image_z_range[0],
                           image_z_range[1],
                           LATERAL_PIXEL_DENSITY_DEFAULT,
                           trans)

In [ ]:
decimation_factor = 1
interpolation_factor = 10

# Low cutoff freq, high cutoff, transition band
filters_params = None

tx_start_del_us = tx_start_delay_us + tr_sw_delay_us

start_time = -tx_start_del_us*10**(-6)
correction_time_shift = 60 * 10 ** (-9)

alpha_fov_apod = 40

SAMPLING_FREQ = fpga.afe_clk
trans.set_active_elements([x for x in range(31)])

In [ ]:
from pybf.scripts.beamformer_cartesian_realtime import BFCartesianRealTime

bf = BFCartesianRealTime(SAMPLING_FREQ,
                         tx_strategy,
                         trans,
                         decimation_factor,
                         interpolation_factor,
                         img_res,
                         img_config,
                         db_range=db_range,
                         start_time=start_time,
                         correction_time_shift=correction_time_shift,
                         alpha_fov_apod=alpha_fov_apod,
                         bp_filter_params=filters_params)

In [ ]:
from pybf.pybf.visualization import plot_image

img_data = bf.beamform(numpy_arr_filt[:, RX_MAPPING,:][11,:31,:])

plot_image(
    np.abs(img_data),
    image_x_range=image_x_range,
    image_z_range=image_z_range,
    db_range=db_range
)